# Compute cost — GPU-hours and API calls, the axis every other contrast lacks  `[EVAL]` `[TRAINING]`

**What this family answers.** Every other contrast in the EDA is indexed by **iteration**, which is
not a fixed unit of spend: a K=5 GRPO optimizer step costs ~1.9× a K=0 step, and a whole PTO
iteration costs a fraction of a GRPO one (its dominant phase is the preference-tree *build*; GRPO's
reward computation happens inside the training loop). This notebook recovers the missing x-axis
from artifacts already on disk — **GPU-hours** per (arm, iteration) reconstructed from artifact
mtimes (`eda_analysis.compute`), and **API calls** per training iteration read from
`generations.jsonl` + the conversation CSVs (`eda_analysis.tails`) — and re-reads the four thesis
contrasts as a function of **budget** under **both graders side by side** (column `judge`:
`gpt-4o-mini` = the training oracle, `claude-haiku-4-5` = the held-out judge). Judge-invariant
family: exports go to `results/compute/cost/{tables,figures}/` with **no `<judge>/` level** — the
grader is a column, never a folder.

Ported from `7_Stats` §4e (the tracked `compute_by_arm / compute_by_iteration / k_step_multiplier /
iso_compute_contrast / budget_sweep` set — same rows for the same arm and grader as the retired
`results/L5/tables/7_stats/<judge>/`) and from the look-ahead paper's promoted `compute_axis.py`
+ `tail_audit.py` (API part) generators — the paper's `tables/compute_axis_*.csv`,
`tail_audit_api_*.csv` are the frozen fixture; the promoted artifacts here drop the script prefix
(`compute_axis_by_arm` → `compute_by_arm`, `tail_audit_api_calls` → `api_calls`).

**Where the hours come from.** `iteration_metadata.json`'s `training_time_s` / `generation_time_s` /
`pref_pair_time_s` are **per-PROCESS**, so a resumed iteration records only its last session
(GRPO_LA5 iteration 1 logs 14,501 s for 7.7 h of steps; PTO logs `pref_pair_time_s = 3.2 s` for a
~30-min build reloaded from `pairs.csv`). Every phase is therefore timed from **artifact mtimes**:
`generate` = mtime span of `model_iter_{k-1}/*.csv`; `build` (PTO only) = last conversation mtime →
`pref_pairs/pairs.csv`; `train` = GRPO `training/completions/*.parquet` (one per step) / PTO
TensorBoard `wall_time`. Deltas outside `(0, 3600 s)` are resume gaps or re-synced Drive mtimes and
are imputed at the phase median (`n_imputed`). `cum_gpu_h` at iteration *k* is the cost of having
produced the policy the score lake calls `<Arm>_I{k}`.

**Conventions (stated again in every caption).**
- **Sign.** K contrasts are computed as `arm_a = LA5, arm_b = LA0`, so `mean_delta = K5 − K0`
  (the tracked-EDA convention; above zero = look-ahead ahead); the paper's `+ ⇒ K=0 higher` reading
  is carried beside it as `delta_K0_minus_K5` / `dz_K0_minus_K5`. Method contrasts: `arm_a = PTO,
  arm_b = GRPO`, `+ mean_delta ⇒ PTO higher`. On `MICI` (and every `MICI_*` channel, lower = better)
  a positive Δ means `arm_a` is *worse*.
- **Pairing.** Iso-compute pairs *different* iterations across arms, so `file_index` pairing is
  invalid (the 96 personas reshuffle `seed + k + 1` every iteration) — everything here pairs on
  `persona_id` (n = 96). Bootstrap CIs use `stats.paired_arrays` seeded with `constants.BOOT_SEED`
  (the paper generators seeded 0, so CI bounds may differ from the fixture in the third decimal;
  means / dz / p / n match exactly).
- **Budget ceilings.** All four arms trained the same **10 iterations**, for very different money —
  each arm's ceiling is its LAST `cum_gpu_h` in `compute_by_iteration` (read the numbers there; they
  move whenever a run advances). So a sweep runs over `arm_a`'s own budget grid, and once a budget
  passes `arm_b`'s ceiling the `arm_b` side stops advancing — its whole run is already affordable at
  that price. Every sweep / iso frame inherits that: an iso-compute row whose `budget_ratio` leaves
  ~0.9–1.1 is not a matched comparison and should not be quoted as one.
- ⚠ **Quote `budget_sweep`, not a single iso-compute row** — the sign of a lever is a function of
  budget.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 60)

import os, eda_analysis
from eda_analysis import exports, plotting, compute, tails, behavior, reliability as R
from eda_analysis.constants import judge_dirname, set_active_judge, PRIMARY_JUDGE_TAG, BOOT_SEED, LOWER_IS_BETTER
cfg = eda_analysis.EdaConfig(family="compute/cost", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # reset_results wiped the banner notebook_setup wrote — re-stamp it

## 0 · Load — both graders, both K arms of both methods, the cost frame  `[EVAL]`
`scores_by_judge` gives the same arm/metric filters under each grader on disk (primary first);
`cross_k_arms` is the arm list (== `S.ARMS` under the all-arms default). `compute.iteration_compute`
reconstructs GPU-hours per (arm, iteration) from artifact mtimes; `compute_by_iteration_with_floor`
adds the generation-time **floor** columns (`gen_h_floor = max(mtime span, recorded
generation_time_s)` — the mtime span starts at the first conversation write, so it misses the first
batch of 64 (~0.1 h/iter) and collapses to ~0 when all CSVs flush together, as for PTO_LA5 iters 1–5
whose time lands in iter 6). The headline `gen_h / gpu_h / cum_gpu_h` stay the tracked numbers; the
floor is a sensitivity column.

In [ ]:
SC = eda_analysis.scores_by_judge(S)          # {'gpt-4o-mini': scores_long, 'claude-haiku-4-5': scores_long}, primary FIRST
JUDGES = list(SC)
PRIMARY = judge_dirname(PRIMARY_JUDGE_TAG)
HELDOUT_TAGS = R.second_judge_tags()
ROLE = {j: ("training oracle" if j == PRIMARY else "held-out judge") for j in JUDGES}
ARMS = eda_analysis.cross_k_arms(S)           # both K arms of both methods (Arm objects, with runs_dir)
ARM_ORDER = [a for a in ["PTO_LA0", "PTO_LA5", "GRPO_LA0", "GRPO_LA5"] if a in {x.label for x in ARMS}]
PAL = plotting.arm_palette(ARM_ORDER)
for j, sc in SC.items():
    print(f"{j:>18} ({ROLE[j]:14s}) scores_long {sc.shape} | arms {sorted(sc.arm.unique())}")
assert len(JUDGES) >= 2, "compute/cost needs the primary AND a held-out judge on disk (both are put side by side)"

COMP = compute.iteration_compute(ARMS)        # mtime-reconstructed GPU-h per (arm, iteration); iteration 0 = 0 h
if COMP.empty:
    raise SystemExit("no run artifacts readable — compute axis skipped (Drive symlinks offline?).")
COMPF = compute.compute_by_iteration_with_floor(COMP, ARMS)   # + gen_h_meta / gen_h_floor / gpu_h_floor / cum_gpu_h_floor
SUMM = compute.compute_by_arm_with_floor(COMPF)                # one row per arm + total_gpu_h_floor / build_share / train_share
print("cost frame:", COMP.shape, "| arms:", ARM_ORDER,
      "| last iters:", {a: int(COMP[COMP.arm == a].iteration.max()) for a in ARM_ORDER})

# Two caption fragments, both DERIVED from the frame in hand so they cannot go stale the way a
# hard-coded censoring sentence did (this notebook's captions asserted "GRPO_LA5 is right-censored"
# for months after that arm finished at iteration 10).
#   SUPPORT      — "" whenever every arm reaches the same iteration; the sentence only when one is short.
#   CEILING_NOTE — compute.CENSOR_NOTE (how to read a sweep's budget grid, always true) plus THIS
#                  render's per-arm ceilings, so the reader never has to trust a number in prose.
SUPPORT = eda_analysis.support_note(COMP, base_col="", label=False, subject="no later iteration on disk")
SUPPORT_S = f"{SUPPORT} " if SUPPORT else ""          # caption-safe: trailing space only when non-empty
CEILINGS = COMP.groupby("arm").agg(cum_gpu_h=("cum_gpu_h", "max"), last_iter=("iteration", "max")).reindex(ARM_ORDER)
CEILING_NOTE = (compute.CENSOR_NOTE + " Ceilings on this render: "
                + "; ".join(f"{a} {r.cum_gpu_h:.3f} GPU-h at I{int(r.last_iter)}" for a, r in CEILINGS.iterrows()) + ".")
print("support note:", repr(SUPPORT), "\nceilings:", CEILINGS.round(3).to_dict())

## 1 · What each arm actually cost  `[TRAINING]`
**Purpose.** THE table that makes "arm X ran N iterations" checkable against what arm X *cost*. One
row per arm — iterations trained, phase GPU-hours (generate / build / train), the total, cost per
iteration and the floor — then the per-(arm, iteration) frame every iso-compute contrast indexes on.
`cost_ratios` shows the arithmetic behind every ratio the write-up quotes (rule: a composite number
shows its arithmetic). Note that all four arms trained the SAME 10 iterations for wildly different
money — and in particular that the two GRPO arms are **not** budget-matched: `51.205 / 27.906 =
1.835×`, i.e. look-ahead is a ~1.8× bill at a matched iteration count. That gap is the whole reason
the iteration axis cannot answer the budget question.

In [ ]:
by_arm_cols = ["arm", "method", "K", "last_iter", "n_iters", "gen_h", "build_h", "train_h", "total_gpu_h",
               "median_step_s", "n_imputed", "train_source", "gpu_h_per_iter", "total_gpu_h_floor",
               "build_share", "train_share"]
BY_ARM = SUMM[[c for c in by_arm_cols if c in SUMM.columns]].reset_index(drop=True)
print("=== GPU-hours per arm (generate + build + train) ==="); display(BY_ARM.round(3))
exports.save_table(BY_ARM, "compute_by_arm", caption=(
    "One row per arm: iterations trained, phase GPU-hours (generate rollouts / build preference trees / train) and "
    "total, cost per iteration (eda_analysis.compute.compute_summary), reconstructed from artifact mtimes because "
    "iteration_metadata.json's timings are per-PROCESS and undercount every resumed iteration. train_source records "
    "which artifact timed the optimizer loop (GRPO: one completions parquet per step; DPO writes none and is timed "
    "from TensorBoard wall_time). n_imputed = intervals replaced by the phase median (resume gap or re-synced Drive "
    "mtime). build_share/train_share = phase / total. total_gpu_h_floor uses gen_h_floor = max(mtime span, recorded "
    "generation_time_s) per iteration (the mtime span misses the first batch, ~0.1 h/iter, and is ~0 for PTO_LA5 "
    "iters 1-5); the headline total_gpu_h is the tracked EDA number. n_iters / last_iter say how far each arm ran and "
    "total_gpu_h what it cost - equal iteration counts are NOT equal budgets, which is the point of this family. "
    f"Judge-invariant (no grader involved). {SUPPORT_S}Same rows as the retired results/L5/tables/7_stats/<judge>/"
    "compute_by_arm; the promoted paper table compute_axis_by_arm."))

by_iter_cols = ["arm", "method", "K", "iteration", "n_steps", "median_step_s", "n_imputed",
                "gen_h", "build_h", "train_h", "gpu_h", "cum_gpu_h", "train_source",
                "gen_h_meta", "gen_h_floor", "cum_gpu_h_floor"]
BY_ITER = COMPF[[c for c in by_iter_cols if c in COMPF.columns]].reset_index(drop=True)
print("=== per (arm, iteration) — head ==="); display(BY_ITER[BY_ITER.iteration > 0].head(12).round(3))
exports.save_table(BY_ITER, "compute_by_iteration", caption=(
    "GPU-hours per (arm, iteration), reconstructed from artifact mtimes by eda_analysis.compute.iteration_compute "
    "(gap_cutoff 3600 s; deltas outside (0, 3600 s) imputed at the phase median, n_imputed counts them). "
    "gen = rollout pass that produced model_iter_{k-1}; build = PTO pref-tree branching + oracle (PTO only); "
    "train = optimizer loop (GRPO: completions parquet mtimes; PTO: TensorBoard wall_time). Iteration 0 = the base "
    "policy (0 h by construction). cum_gpu_h = cost of having produced <Arm>_I{k} - the policy the score lake calls "
    "that - so it joins directly onto scores_long (headline = the tracked EDA numbers). gen_h_meta = "
    "iteration_metadata.json generation_time_s/3600 (per-PROCESS); gen_h_floor = max(gen_h, gen_h_meta) is a FLOOR "
    "on generation time (the mtime span starts at the first conversation write, so it misses the first batch of 64, "
    "~0.1 h, and collapses to ~0 when all CSVs flush together: PTO_LA5 iters 1-5, whose time lands in iter 6); "
    "cum_gpu_h_floor re-cumulates with it. Each arm's rows run to its own last trained iteration and its last "
    f"cum_gpu_h is that arm's budget ceiling - read both off the iteration column. Judge-invariant. {SUPPORT_S}"
    "Iteration>0 rows = the retired results/L5/tables/7_stats/<judge>/compute_by_iteration; the promoted paper table "
    "compute_axis_by_iteration."))

RATIOS = compute.cost_ratios(SUMM)
print("=== cost ratios with their arithmetic (headline + generation-floor variants) ==="); display(RATIOS.round(3))

## 2 · The per-iteration price of look-ahead  `[TRAINING]`
**Purpose.** What one optimizer step costs at K=5 vs K=0, per iteration and NOT pooled. GRPO's
look-ahead cost lands INSIDE the optimizer step (5 extra simulated turns per candidate), so the
median step-seconds ratio is the right unit there; PTO's DPO step carries no look-ahead (ratio ~1) —
its look-ahead cost lands in the pref-tree BUILD phase, so the build-hour ratio and the whole-iteration
ratio are shown instead. ⚠ Iteration 1 of GRPO_LA5 ran at `LOOKAHEAD_SUB_BATCH_SIZE=64` with a fat
API-latency tail (ratio 2.41): quote the settled iterations 3–10 off the table itself (median 1.92
across those eight rows, i.e. the ~1.9× that is the physics of 5 extra simulated turns per candidate).

In [ ]:
SM = compute.step_multiplier_table(COMP)
if SM.empty:
    print("no GRPO step data — step multiplier skipped.")
else:
    print("=== per-step / per-build / per-iteration cost of look-ahead, by iteration ==="); display(SM.round(3))
    exports.save_table(SM, "step_multiplier", caption=(
        "The per-iteration price of look-ahead. GRPO: median optimizer-step seconds K=0 vs K=5 and their ratio "
        "(eda_analysis.compute.step_multiplier; the K=5 reward computation runs 5 extra simulated turns per candidate "
        "INSIDE the training loop). A row exists for every iteration both K arms of that method trained. PTO: the DPO "
        "step carries no look-ahead (ratio ~1); PTO's look-ahead cost lands in the pref-tree BUILD phase, so its "
        "build_h ratio and whole-iteration gpu_h ratio are shown instead. Reported per iteration and NOT pooled: "
        "iteration 1 of GRPO_LA5 ran at LOOKAHEAD_SUB_BATCH_SIZE=64 with a fat API-latency tail (ratio 2.41), so quote "
        "the settled iterations 3 onwards (~1.9x) and read them off this table rather than from prose. Judge-invariant. "
        "The GRPO columns are the retired results/L5/tables/7_stats/<judge>/k_step_multiplier (median_s_K0/K5, "
        "ratio_median); the promoted paper table compute_axis_step_multiplier."))

## 3 · Iso-compute contrasts — every arm pair at matched cumulative GPU-hours  `[EVAL]`
**Purpose.** The tracked fixed-endpoint form of the budget question: for every trained iteration
of `arm_a` the closest-budget iteration of `arm_b`, all rubrics, persona-paired Wilcoxon + dz + Holm
across rubrics within each budget-matched pair — computed under **each grader** (column `judge`,
primary first). `budget_ratio = arm_b's spend / arm_a's`; anything outside ~0.9–1.1 is not an
iso-compute comparison and should not be quoted as one. Read the endpoint rows below as a preview
only — §4's sweep is the honest form (this table freezes each arm at one iteration).

In [ ]:
ISO_PAIRS = [(a, b) for _tag, a, b, _lab in compute.CONTRASTS]      # (LA5, LA0) per method; (PTO, GRPO) per K
frames = []
for j, sc in SC.items():
    for a, b in ISO_PAIRS:
        if not {a, b} <= set(sc.arm.unique()):
            continue
        T = compute.iso_compute_contrast(sc, COMP, a, b)
        if not T.empty:
            frames.append(T.assign(judge=j))
if not frames:
    ISO = pd.DataFrame(); print("no iso-compute pairs available.")
else:
    ISO = pd.concat(frames, ignore_index=True)
    ISO["judge"] = pd.Categorical(ISO["judge"], categories=JUDGES, ordered=True)
    ISO = ISO.sort_values(["judge", "arm_a", "arm_b", "iter_a", "metric"], kind="stable").reset_index(drop=True)
    ISO["judge"] = ISO["judge"].astype(str)
    iso_view = ISO[["judge", "arm_a", "arm_b", "iter_a", "iter_b", "cum_gpu_h_a", "cum_gpu_h_b", "budget_ratio",
                    "metric", "n", "mean_delta", "dz", "p", "p_holm"]]
    print("=== matched-BUDGET contrasts (+ => arm_a higher) - endpoint rows, both graders ===")
    display(iso_view[iso_view.metric.isin(["Q1Q2", "MICI", "MITI"])
                     & (iso_view.groupby(["judge", "arm_a", "arm_b"]).iter_a.transform("max") == iso_view.iter_a)].round(4))
    exports.save_table(iso_view, "iso_compute_contrast", caption=(
        "Every arm pair contrasted at MATCHED CUMULATIVE GPU-HOURS rather than matched iteration, BOTH graders in one "
        "table (column `judge`: gpt-4o-mini = the training oracle, claude-haiku-4-5 = the held-out judge), persona-paired "
        "(n = 96 personas; iso-compute reads a DIFFERENT iteration from each arm, so file_index pairing would join "
        "unrelated conversations - personas reshuffle seed+k+1 every iteration). For every trained iteration of arm_a "
        "the closest-budget iteration of arm_b; budget_ratio = arm_b's spend / arm_a's - anything outside ~0.9-1.1 is "
        "not an iso-compute comparison and should not be quoted as one. Every arm_a iteration gets a row, including "
        "those whose budget exceeds arm_b's WHOLE run: there the pair falls back on arm_b's last iteration and "
        "budget_ratio drops well below 0.9, so read the ratio before quoting a row. + mean_delta => arm_a higher: K "
        "contrasts have arm_a = the K=5 arm (mean_delta = K5 - K0; the paper's + => K=0 higher reading is the "
        "negative), method contrasts have arm_a = PTO; on MICI (lower = better) a positive delta means arm_a is WORSE. "
        f"Wilcoxon + dz + Holm across rubrics within each budget-matched pair. {SUPPORT_S}A fixed-endpoint contrast "
        "freezes each arm at one iteration - quote the budget_sweep tables instead. Same rows for the same grader as "
        "the retired results/L5/tables/7_stats/<judge>/iso_compute_contrast."))

## 4 · Budget sweeps — the lever's sign as a function of spend  `[EVAL]`
**Purpose.** The honest form of the iso-compute question. At each of `arm_a`'s cumulative GPU-h
budgets both arms are represented by the **best checkpoint they could have reached for that money**
(best on `select_metric` under the named grader; `MICI` selects the LOWEST), and the contrast is
scored on `eval_metric`, persona-paired with a bootstrap 95% CI, Wilcoxon *p* and Holm over the
unique checkpoint pairs of the table. Three variants stack in each table: `Q1Q2 → Q1Q2` (the tracked
sweep, row-for-row `compute.budget_sweep`), `MICI → MICI`, and `Q1Q2 → MICI` (does the
reward-selected policy carry the hack?). One table per contrast × grader (4 × 2 = 8):
`budget_sweep_<contrast>_<judge>`. `budget_sweep_top` is the top-of-sweep verdict with a
`sign_flips_within_sweep` flag — when it is True, quote the curve, not the endpoint.

In [ ]:
SWEEPS = compute.all_budget_sweeps(SC, COMP)        # {(contrast_tag, judge_label): DF}; asserts Q1Q2->Q1Q2 == budget_sweep
CONTRAST_LABEL = {tag: lab for tag, _a, _b, lab in compute.CONTRASTS}
for (tag, jl), df in SWEEPS.items():
    a, b = df.arm_a.iloc[0], df.arm_b.iloc[0]
    exports.save_table(df, f"budget_sweep_{tag}_{jl}", caption=(
        f"Budget sweep, {CONTRAST_LABEL[tag]} ({a} = arm_a vs {b} = arm_b), grader = {jl} ({ROLE.get(jl, '')}). At "
        "each of arm_a's cumulative GPU-h budgets both arms are represented by the best checkpoint they could have "
        "reached for that money (best on select_metric under this grader; MICI selects the LOWEST), and the contrast is "
        "scored on eval_metric paired on persona_id (n = 96 personas; bootstrap 95% CI seeded with BOOT_SEED; Wilcoxon "
        "p; Holm within this table's (select_metric, eval_metric) family over the unique checkpoint pairs). "
        f"{compute.sign_note(tag)} MICI is lower-is-better. Rows select_metric=Q1Q2 -> eval_metric=MICI score the "
        "Q1Q2-selected checkpoints on MICI (does the reward-selected policy carry the hack?). The Q1Q2->Q1Q2 rows "
        "mirror eda_analysis.compute.budget_sweep row-for-row (= the retired results/L5/tables/7_stats/<judge>/"
        f"budget_sweep for this grader). {CEILING_NOTE} {SUPPORT_S}Promoted paper table "
        f"compute_axis_budget_sweep_{tag}_{jl}."))
print(f"{len(SWEEPS)} sweep tables saved:", sorted(f"budget_sweep_{t}_{j}" for t, j in SWEEPS))

TOP = compute.budget_sweep_top(SWEEPS)              # last row of the Q1Q2->Q1Q2 variant per (contrast, judge)
print("=== top-of-sweep verdicts on Q1Q2 (arm_a's LAST budget; + => arm_a higher) ===")
display(TOP.round(4))
tag, jl = ("GRPO_K", PRIMARY)
if (tag, jl) in SWEEPS:
    d = SWEEPS[(tag, jl)]
    print(f"=== {CONTRAST_LABEL[tag]} under {jl}, Q1Q2->Q1Q2 (mean_delta = K5 - K0) ===")
    display(d[(d.select_metric == "Q1Q2") & (d.eval_metric == "Q1Q2")]
            [["budget_gpu_h", "best_iter_a", "best_iter_b", "mean_a", "mean_b", "n", "mean_delta", "dz", "ci_lo", "ci_hi", "p", "p_holm"]].round(4))

### 4b · The look-ahead sweep as a picture — both graders × both methods  `[EVAL]`
Rows = grader, cols = method; x = the K=5 arm's cumulative GPU-h; y = the paired Q1+Q2 delta
**K5 − K0** between the best-within-budget checkpoints (above zero = look-ahead ahead — the
tracked-EDA sign, NOT the paper's `+ ⇒ K=0 higher` table convention), bootstrap 95% CI bars, hollow
markers = Holm *p* ≥ .05, labels `I<K5>/I<K0>` = the selected iterations. The GRPO curve crosses
zero: at small budgets look-ahead is clearly WORSE (it buys fewer iterations for the money — the
significant −0.569 / −0.495 trough sits at ~13 GPU-h), the sign turns over somewhere around
23–35 GPU-h, and by the top of the sweep it is significantly POSITIVE on both graders (+0.435 at
51.2 GPU-h under the training oracle, +0.275 under the held-out judge; Holm *p* < .001 both). The
crossing, not either endpoint, is the artifact to quote.

In [ ]:
fig = plotting.budget_sweep_grid(SWEEPS, methods=("PTO", "GRPO"), judges=JUDGES)
if fig is not None:
    exports.save_fig(fig, "budget_sweep", caption=(
        "The look-ahead lever vs budget, 2x2 (rows = grader: top gpt-4o-mini = the training oracle, bottom claude-"
        "haiku-4-5 = the held-out judge; cols = method): x = the K=5 arm's cumulative GPU-h; y = paired Q1+Q2 delta "
        "K5 - K0 between the best-within-budget checkpoints (mean_delta as tabled in budget_sweep_<method>_K_<judge>, "
        "Q1Q2->Q1Q2 rows; above zero = look-ahead ahead - the tracked-EDA sign, the paper's + => K=0 higher is the "
        "negative), bootstrap 95% CI bars (BOOT_SEED), hollow = Holm p >= .05, labels I<K5>/I<K0> = the selected "
        f"iterations; persona_id pairing, n = 96. {CEILING_NOTE} {SUPPORT_S}The curve, not any single point, is the "
        "artifact to quote: the GRPO panels cross zero - look-ahead buys fewer iterations for the money at small "
        "budgets and only pays back at the large ones. Was the retired "
        "results/L5/figures/7_stats/<judge>/budget_sweep (GRPO only, one grader); the promoted paper figure "
        "compute_axis_fig_budget_sweep."))
    plt.show()

## 5 · Cross-judge selection — is the verdict a selection artefact?  `[EVAL]`
**Purpose.** Same-judge best-within-budget selection is optimistic for the selecting grader. Here
each arm's checkpoint is SELECTED on `select_judge`'s Q1+Q2 means and the paired contrast is SCORED
on `eval_judge`'s Q1+Q2, for every (select, eval) combination; `honest_selection` = the grader that
picked the checkpoint is not the grader that scores it (the same-judge rows reproduce §4). A verdict
that holds only when the same grader selects and scores is a selection artefact — the
`honest_selection` rows of the verdict table are the ones to quote.

In [ ]:
XJ = compute.budget_sweep_crossjudge(SC, COMP, metric="Q1Q2")
VERD = compute.crossjudge_verdicts(XJ, alpha=0.05)
if XJ.empty:
    print("cross-judge sweep empty (one grader only?).")
else:
    exports.save_table(XJ, "budget_sweep_crossjudge", caption=(
        "Cross-judge selection sweep on Q1+Q2, all four contrasts. Each arm's best-within-budget checkpoint is SELECTED "
        "on select_judge's Q1Q2 means and the paired contrast is SCORED on eval_judge's Q1Q2 (persona_id pairing, "
        "n = 96; bootstrap 95% CI, BOOT_SEED; Wilcoxon p; Holm within each (contrast, select_judge, eval_judge) family "
        "over unique checkpoint pairs). honest_selection = the grader that picked the checkpoint is not the grader that "
        "scores it (the same-judge rows reproduce the budget_sweep_<contrast>_<judge> tables). "
        f"{compute.SIGN_K} {compute.SIGN_M} delta_K0_minus_K5 is blank for method contrasts. {CEILING_NOTE} "
        f"{SUPPORT_S}Promoted paper table compute_axis_budget_sweep_crossjudge."))
    exports.save_table(VERD, "budget_sweep_crossjudge_verdicts", caption=(
        "Top-of-sweep verdicts (each contrast at arm_a's LAST cumulative budget) under every (select_judge, eval_judge) "
        "combination, from budget_sweep_crossjudge; verdict uses p_holm < 0.05. A verdict that holds only when the same "
        "grader selects and scores is a selection artefact; the honest_selection rows are the ones to quote. "
        f"+ mean_delta => arm_a higher ({compute.SIGN_K} {compute.SIGN_M}) Paired on persona_id, n = 96; p_holm within "
        f"the (contrast, select_judge, eval_judge) family. {CEILING_NOTE} At arm_a's top budget the best_iter_b column "
        "therefore often names arm_b's own last affordable checkpoint rather than an equal-cost one - read best_iter_a "
        f"/ best_iter_b, not just the verdict. {SUPPORT_S}Promoted paper table "
        "compute_axis_budget_sweep_crossjudge_verdicts."))
    print("=== verdicts at the top budget - every (select_judge, eval_judge) combination ===")
    display(VERD[["contrast", "select_judge", "eval_judge", "honest_selection", "budget_gpu_h", "best_iter_a", "best_iter_b",
                  "mean_delta", "dz", "ci_lo", "ci_hi", "p_holm", "verdict"]].round(4))

## 6 · Look-ahead at matched compute on the behaviour channels  `[EVAL]`
**Purpose.** The K contrast at matched GPU-hours on the *behaviour* channels rather than the rubrics
— over-praise / advise-without-permission per session and per therapist turn, MI-inconsistent acts
per session (`MICI_*`, lower = better), MITI-coded affirmations (`B6_AF`, per session / per turn) and
the deterministic text measures `conv_len` (utterances) / `mean_turn_len` (chars, no valence,
grader-independent, reported ONCE under `judge = "text (grader-independent)"`). The oracle-coded
channels are judge-DEPENDENT, so the channel frame is loaded once per grader by switching the active
judge (`behavior.channel_scores_long` has no `scores_by_judge` twin). Two tables: `iso_channels` —
every trained K=5 iteration vs the closest-budget K=0 iteration of the same method (`iso_ok` flags
0.9–1.1; the K=0 arm is the cheaper one in both methods, so past its ceiling — PTO_LA0 ~8.12 GPU-h,
GRPO_LA0 ~27.91 — the late K=5 iterations have no iso partner and are flagged False: PTO_LA5 from
iter 5 on, GRPO_LA5 from iter 7 on) — and `iso_channels_selected` — the channels at the checkpoints
an operator would actually DEPLOY (the best-within-budget K=5 / K=0 checkpoints at the top budget of
§4's Q1Q2 sweep). `mean_delta = K5 − K0` in the channel's own unit; Holm within the channel family
per budget pair. Never averaged across graders.

In [ ]:
CH = {}
for j in JUDGES:                                   # oracle-coded channels are read from THAT judge's MITI/MICI partition
    tag = "" if j == PRIMARY else next(t for t in HELDOUT_TAGS if judge_dirname(t) == j)
    set_active_judge(tag, 0)
    CH[j] = behavior.channel_scores_long(ARMS)
set_active_judge("", 0)                            # leave the process on the primary grader
for j in CH:
    CH[j] = CH[j][CH[j]["questionnaire"].isin(compute.CHANNELS)].copy()
    print(f"{j:>18}: channel frame {CH[j].shape} | channels {sorted(CH[j].questionnaire.unique())}")

ISO_CH = compute.iso_channels(CH, COMP)
SEL_CH = compute.iso_channels_selected(CH, SWEEPS)
if ISO_CH.empty:
    print("iso_channels empty.")
else:
    exports.save_table(ISO_CH, "iso_channels", caption=(
        "Look-ahead at MATCHED compute on the behaviour channels (eda_analysis.compute.iso_compute_contrast on "
        "behavior.channel_scores_long, both graders side by side in column `judge`). For every trained iteration of the "
        "K=5 arm (arm_a) the K=0 iteration of the same method with the closest cumulative GPU-h is paired on persona_id "
        "(n = 96; budget_ratio = b/a; iso_ok flags 0.9-1.1). The K=0 arm is the cheaper one in both methods, so past its "
        "ceiling (PTO_LA0 8.12 GPU-h, GRPO_LA0 27.91) the late K=5 iterations have no iso partner and are flagged False "
        "- PTO_LA5 from iter 5 on, GRPO_LA5 from iter 7 on; quote iso_ok rows only. mean_delta = K5 - K0 on the "
        "channel's own unit; delta_K0_minus_K5 = -mean_delta is the paper's convention (+ => K=0 higher). direction "
        "says how to read the sign: MICI_* channels are lower-is-better (over-praise / advise-without-permission per "
        "session and per therapist turn, MI-inconsistent acts per session); B6_AF = MITI-coded affirmations per session "
        "/ per turn; conv_len (utterances) and mean_turn_len (chars) are deterministic text measures with no valence "
        "and are grader-independent (reported once, judge = 'text (grader-independent)'). Bootstrap 95% CI (BOOT_SEED); "
        "Holm within the 9-channel family at one budget pair. MICI/B6_AF rows are shown under both graders side by "
        f"side, never averaged. {SUPPORT_S}Promoted paper table compute_axis_iso_channels."))
    print("=== channels at matched compute - iso_ok rows, top budget per (contrast, judge) ===")
    ok = ISO_CH[ISO_CH.iso_ok]
    display(ok[ok.groupby(["contrast", "judge"]).iter_a.transform("max") == ok.iter_a]
            [["contrast", "judge", "channel", "direction", "iter_a", "iter_b", "budget_ratio", "n", "mean_delta", "dz",
              "ci_lo", "ci_hi", "p_holm"]].round(3))
if SEL_CH.empty:
    print("iso_channels_selected empty.")
else:
    exports.save_table(SEL_CH, "iso_channels_selected", caption=(
        "Behaviour channels at the checkpoints an operator would actually deploy: for each method, the K=5 (arm_a) and "
        "K=0 (arm_b) checkpoints selected as best-within-budget on Q1Q2 under the named grader at the TOP budget of the "
        "K sweep (the last Q1Q2->Q1Q2 row of budget_sweep_<method>_K_<judge>), contrasted on each channel paired on "
        "persona_id (n = 96; bootstrap 95% CI, BOOT_SEED; Wilcoxon p; Holm within the channel family per (contrast, "
        "judge)). At that top budget the whole K=0 run is affordable, so iter_b is that arm's best checkpoint outright "
        "rather than an equal-cost one - this is the deployment question, not a matched-cost one. mean_delta = K5 - K0 "
        "in the channel's unit; delta_K0_minus_K5 is the paper's convention (+ => K=0 higher); direction gives the "
        "valence (MICI_* lower=better; B6_AF higher = more MI-consistent; text channels none, grader-independent, "
        f"reported once). Both graders side by side, never averaged. {SUPPORT_S}Promoted paper table "
        "compute_axis_iso_channels_selected."))
    print("=== channels at the DEPLOYED checkpoints ===")
    display(SEL_CH[["contrast", "judge", "iter_a", "iter_b", "channel", "direction", "mean_a", "mean_b", "mean_delta", "dz",
                    "ci_lo", "ci_hi", "p_holm"]].round(3))

## 7 · The compute axis as pictures  `[EVAL]` `[TRAINING]`
- **`compute_trajectory`** — THE figure this EDA was missing: Q1+Q2 (mean ± SEM over the 96
  personas) against **cumulative GPU-hours** instead of iteration, four arms, one panel per grader,
  iteration 0 at 0 h, the last point labelled with its iteration. Unequal marker spacing along x IS
  the finding — an arm whose iterations are cheap gets many markers close together, and four arms
  that all ran the SAME 10 iterations end at wildly different x (8.1 / 19.7 / 27.9 / 51.2 GPU-h).
  `compute_trajectory_col` is the SAME two panels stacked for a single column (same data, same
  style; layout is the only difference).
- **`cost_breakdown`** — where each arm's GPU-hours go per iteration, stacked generate / build /
  train, one panel per arm. Explains the cost gap rather than asserting it: PTO's dominant phase is
  the preference-tree build (absent in GRPO, whose reward computation happens inside the training
  loop). The one known artefact is annotated (PTO_LA5's I1–I5 generation landing in I6 because its
  conversation mtimes were batch-flushed); the right-censoring label is derived per arm, so none is
  drawn while every arm ends at the same iteration — as they all do now.

In [ ]:
fig = plotting.trajectory_by_compute(SC, COMP, metric="Q1Q2", arms=ARM_ORDER, layout="wide")
if fig is not None:
    exports.save_fig(fig, "compute_trajectory", caption=(
        "Q1+Q2 (mean +/- SEM over the 96 personas) vs cumulative GPU-hours instead of iteration, four arms, one panel per "
        "grader (left gpt-4o-mini = the training oracle, right claude-haiku-4-5 = the held-out judge); iteration 0 at "
        "0 h; K=0 solid/circle, K=5 dashed/square; last point labelled with its iteration. Unequal marker spacing along "
        "x IS the finding: an arm whose iterations are cheap gets many markers close together, and four arms that all "
        "ran the same 10 iterations end at wildly different x (read each arm's last cum_gpu_h off compute_by_arm). "
        f"{SUPPORT_S}Hours are mtime-reconstructed (compute_by_iteration.cum_gpu_h). Was the retired "
        "results/L5/figures/7_stats/<judge>/compute_trajectory (one grader); the promoted paper figure "
        "compute_axis_fig_trajectory."))
    plt.show()
fig = plotting.trajectory_by_compute(SC, COMP, metric="Q1Q2", arms=ARM_ORDER, layout="col")
if fig is not None:
    exports.save_fig(fig, "compute_trajectory_col", caption=(
        "Single-column variant of compute_trajectory: the SAME two grader panels (top gpt-4o-mini = the training oracle, "
        "bottom claude-haiku-4-5 = the held-out judge) stacked with shared x and y, sized for a 3.4-in column. Same data, "
        "same style, same support; the layout is the only difference. Promoted paper figure "
        "compute_axis_fig_trajectory_col."))
    plt.show()

fig = plotting.cost_breakdown(COMP, arms=ARM_ORDER)     # iteration frame -> per-iteration stacked panels (the paper's fig_breakdown)
if fig is not None:
    exports.save_fig(fig, "cost_breakdown", caption=(
        "Where each arm's GPU-hours go, per iteration: stacked generate (white + arm-colour hatch) / build (arm colour, "
        "light; PTO only) / train (solid arm colour), one panel per arm, panel title = the arm's total. Explains the cost "
        "gap rather than asserting it: PTO's dominant phase is the preference-tree build (absent in GRPO, whose reward "
        "computation happens inside the training loop), which is why per-step timings alone cannot compare the two "
        "methods. Known artefact annotated: PTO_LA5's generation of I1-I5 lands in I6 (batch-flushed conversation "
        "mtimes; cumulative totals are right, per-iteration gen splits are not, for that arm). The right-censoring label "
        f"is derived per arm and is drawn only for an arm that ends before the others. {SUPPORT_S}Judge-invariant. Was "
        "the retired results/L5/figures/7_stats/<judge>/cost_breakdown (per-arm stacks); the promoted paper figure "
        "compute_axis_fig_breakdown."))
    plt.show()

## 8 · The API axis — oracle and patient-simulator calls per training iteration  `[TRAINING]`
**Purpose.** GPU-hours are not the only bill: the oracle (Q1 + Q2 per scored candidate) and the
patient simulator (eval convs, PTO trunk replies, and the realized patient turns inside K=5 tails)
are API calls, and the binding cost constraint of the project. `tails.api_calls` accounts them per
arm × training iteration *n* — a row is the cost of iteration *n*: the 96 eval convs generated at
its start by π_{n−1} (`eval_convs_of = model_iter_{n-1}`) PLUS the training-time calls; the last row
per arm (`row_kind = 'final eval pass'`) is the post-loop generate-only pass with no training.
GRPO's oracle calls are read from `generations.jsonl` and RESCALED to the ground-truth step count
(`log_coverage` < 1 where a crashed iteration's pre-resume records were lost). `api_ratio` sums K=5 /
K=0 over the matched iterations per method with the arithmetic shown. ⚠ The oracle ratio is NOT 1
even though calls per candidate are matched: the number of candidates per iteration differs between
arms (GRPO's prompt count follows the eval-conv length of the current policy; PTO's branch points
follow how far its trunks grow before the patient closes the session). Judge-invariant — these are
counts of calls, not scores. The tail *audit* itself (what those K=5 tails contain) is owned by
`lookahead/mechanism`.

In [ ]:
API = tails.api_calls(ARMS, verbose=True)          # ~2 min cold: streams every generations.jsonl + reads the eval CSVs
RATIO = tails.api_ratio(API)
if API.empty:
    print("api_calls empty - no generations.jsonl readable.")
else:
    exports.save_table(API, "api_calls", caption=(
        "API-call accounting per arm x training iteration n (row = the cost of iteration n: the 96 eval convs generated "
        "at its start by policy pi_{n-1} = model_iter_{n-1}, PLUS the training-time calls; the last row 'final eval pass' "
        "is the post-loop generate-only pass with no training). oracle_calls_train = Q1 + Q2 calls per scored candidate "
        "(2 per candidate, + recorded retries; GRPO's TRL eval-phase groups included, n_candidates_eval_phase), read from "
        "generations.jsonl and - for GRPO - rescaled to the ground-truth step count (n_steps = training/completions/"
        "*.parquet files, 128 candidates = 16 groups x G=8 per step; log_coverage = logged / expected groups, < 1 where a "
        "crashed iteration's pre-resume records were lost - read which iterations off the log_coverage column itself); "
        "oracle_input_Mchars = the chars the oracle read (prefix + completion + tail, x2 rubrics) as a token proxy. "
        "eval_scoring_calls_run_eval = 96 x 8 instruments per model state (Run_Eval, identical for every arm; per "
        "grader). patient_calls_eval_convs = patient turns in the model_iter_{n-1} CSVs; patient_calls_trunk = PTO "
        "greedy trunk replies (<= 1 per branch point, upper bound within 96); patient_calls_tail = realized patient "
        "turns inside K=5 tails (ceil(realized_turns/2); 1 for a zero-turn tail whose first patient call was made); "
        "patient_calls_total = the sum of the three. therapist_gens_tail = therapist turns generated inside tails (GPU, "
        f"not API). Judge-invariant (call counts, not scores). {SUPPORT_S}Promoted paper table tail_audit_api_calls."))
    exports.save_table(RATIO, "api_ratio", caption=(
        "K=5 / K=0 API-call ratios per method, summed over the matched training iterations named in 'iters'. Two "
        "windows per method: 'iters 1-5 (matched)' is a FIXED early window kept as-is so its numbers stay comparable "
        "with the frozen paper fixture, and 'all matched iters' is every iteration both K arms of that method trained. "
        "Sums are the columns of api_calls.md over those rows (arithmetic shown; total_api_calls = oracle_calls_train + "
        "patient_calls_total); the final patient_calls_tail_per_candidate row per method is the physics (3 patient "
        "calls per full K=5 tail). The K5/K0 ratio for the oracle is NOT 1 even though calls per candidate are matched: "
        "the number of candidates per iteration differs between arms (GRPO's prompt count follows the eval-conv length "
        "of the current policy; PTO's branch points follow how far its trunks grow before the patient closes the "
        "session). Judge-invariant. Promoted paper table tail_audit_api_ratio."))
    print("=== API calls per arm x training iteration (head) ===")
    display(API[["arm", "train_iter", "row_kind", "n_steps", "log_coverage", "n_candidates", "oracle_calls_train",
                 "oracle_input_Mchars", "patient_calls_eval_convs", "patient_calls_trunk", "patient_calls_tail",
                 "patient_calls_total"]].round(2))
    print("=== K5 / K0 ratios (matched iterations) ==="); display(RATIO.round(3))
    fig = plotting.api_calls_fig(API, arms=ARM_ORDER, palette=PAL)
    if fig is not None:
        exports.save_fig(fig, "api_calls", caption=(
            "API calls per training iteration, one line per arm (K=0 solid/circle, K=5 dashed/square), log y: (a) oracle "
            "calls for the training reward (Q1 + Q2 per scored candidate; GRPO rescaled to its true step count), (b) "
            "patient-simulator calls (eval convs of pi_{n-1} + PTO trunk replies + K=5 tail turns). row_kind = "
            "'iteration' rows only (the final eval pass is not a training iteration). Judge-invariant (counts, not "
            f"scores). {SUPPORT_S}Promoted paper figure tail_audit_fig_api."))
        plt.show()

## 9 · Number ledger  `[EVAL]` `[TRAINING]`
Every number the write-up may quote from this family, as citable keys in
`results/compute/cost/tables/compute_numbers.json` — `compute.compute_numbers` (ratios with their
arithmetic, per-arm / per-iteration hours, step multipliers, every sweep row + top verdicts,
cross-judge verdicts, channels at matched compute / at the deployed checkpoints, caveats; the same
keys as the paper's frozen `out/compute_axis.json`, sources re-pointed at this family's tables) plus
the API part of `tails.tails_numbers` (`api.*`, `api_ratio.*`, `api_totals.*` — the tail-audit keys
belong to `lookahead/mechanism`) and the `figures.*` entries. Nothing is computed here that is not
in a table above.

In [ ]:
NUM = compute.compute_numbers(COMPF, SUMM, SM, SWEEPS, XJ, VERD, ISO_CH, SEL_CH,
                              source_prefix="tables/", name_prefix="")
# compute_numbers names the two cost tables `by_arm` / `by_iteration`; this family keeps the tracked EDA names
# (compute_by_arm / compute_by_iteration), so re-point those two sources at the files actually saved.
_REPOINT = {"tables/by_arm.md": "tables/compute_by_arm.md", "tables/by_iteration.md": "tables/compute_by_iteration.md"}
for k, v in NUM.items():
    src = v.get("source", "")
    for a, b in _REPOINT.items():
        if src.startswith(a):
            v["source"] = b + src[len(a):]
# Censoring is a FRAME FACT, not a constant - this ledger shipped "GRPO_LA5 is right-censored" for
# months after that arm finished at iteration 10. Drop any frozen censoring line compute.CAVEATS
# still carries and add the DERIVED one, which is empty while every arm reaches the same iteration.
if "caveats" in NUM:
    _cav = [c for c in NUM["caveats"]["value"] if "right-censored" not in c]
    if SUPPORT:
        _cav.append(SUPPORT)
    NUM["caveats"] = dict(NUM["caveats"], value=_cav)
# The API part of tails_numbers: pass an EMPTY audit so only api.* / api_ratio.* / api_totals.* (+ the pooled
# session-end cross-check, dropped) are built - the tail-audit keys are owned by lookahead/mechanism.
if not API.empty:
    _empty = tails.TailAudit(pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame())
    TN = tails.tails_numbers(ARMS, audit=_empty, api=API, ratio=RATIO, verbose=False)
    for k, v in TN.items():
        if k.startswith(("api.", "api_ratio.", "api_totals.")):
            v = dict(v)   # re-point the source at the files THIS family saved (prefix dropped: api_calls.md / api_ratio.md)
            v["source"] = v["source"].replace("compute/cost/tables/tail_audit_api_", "tables/api_")
            NUM[k] = v
NUM["figures.budget_sweep"] = {"value": "figures/budget_sweep.png", "source": "figures/", "note":
    "2x2 (rows = grader, cols = method): x = K=5 arm's cumulative GPU-h; y = paired Q1Q2 delta K5 - K0 between the "
    "best-within-budget checkpoints (bootstrap 95% CI; hollow = Holm p>=0.05); labels I_K5/I_K0 = selected iterations; "
    "persona_id pairing; the GRPO panels cross zero, so quote the curve and not an endpoint."}
NUM["figures.trajectory"] = {"value": "figures/compute_trajectory.png", "source": "figures/", "note":
    "Q1Q2 mean +- SEM (96 personas) vs cumulative GPU-h per arm, one panel per grader; iteration 0 at 0 h; K=0 solid/"
    "circle, K=5 dashed/square; last point labelled with its iteration - equal iteration counts, very unequal x. "
    "_col = same panels stacked."}
NUM["figures.breakdown"] = {"value": "figures/cost_breakdown.png", "source": "figures/", "note":
    "Stacked GPU-h per iteration per arm (generate / build / train, mtime-reconstructed); PTO_LA5 gen for iters 1-5 "
    "lands in iter 6 (batch-flushed conv mtimes); the right-censoring label is derived per arm and is drawn only for "
    "an arm that ends before the others."}
NUM["figures.api_calls"] = {"value": "figures/api_calls.png", "source": "figures/", "note":
    "API calls per training iteration per arm, log y: (a) oracle calls (training reward), (b) patient-simulator calls."}
NUM["meta.sign"] = {"value": f"{compute.SIGN_K} {compute.SIGN_M} MICI and MICI_* channels are lower = better.", "source": "", "note": ""}
NUM["meta.pairing"] = {"value": "persona-paired on persona_id (never file_index), n = 96 personas; bootstrap CIs seeded with "
                       f"BOOT_SEED={BOOT_SEED}", "source": "", "note": ""}
# DERIVED, never asserted: support_note() is EMPTY unless an arm really IS short relative to the
# others on THIS frame (an in-flight iteration still bills its mtimes into it), so this key cannot
# claim a censoring the data does not have. The budget-ceiling half is true either way.
NUM["meta.censoring"] = {
    "value": (SUPPORT or "no arm is short in this frame: every arm runs to the same last trained iteration.")
             + " " + CEILING_NOTE,
    "source": "tables/compute_by_iteration.md", "note": ""}
path = exports.save_numbers("compute_numbers", NUM, caption=(
    "Number ledger for the compute axis: cost ratios with their arithmetic (ratio.*, ratio_floor.*), per-arm and "
    "per-iteration GPU-hours (by_arm.*, by_iteration.*), step multipliers (step_multiplier.*), every budget-sweep row and "
    "top-of-sweep verdict (sweep.*, sweep_top.*), cross-judge verdicts (crossjudge*.*), channels at matched compute and at "
    "the deployed checkpoints (iso_channels*.*), the API-call rows / K5-over-K0 ratios / per-arm totals (api.*, "
    "api_ratio.*, api_totals.*), the figures and the caveats - each key citing the table it was read from. K contrasts: "
    "mean_delta = K5 - K0 with delta_K0_minus_K5 beside it; method contrasts + => PTO higher; persona-paired n = 96; "
    "equal iteration counts are NOT equal budgets - how far each arm ran and what it cost is in the meta.censoring key, "
    "DERIVED from the frame rather than asserted here."))
print(f"{len(NUM)} keys ->", path)
# every table a source cites must exist in THIS family's tables/ (the ledger is only citable if it points at real files)
import re as _re
_cited = sorted({m.group(1) for v in NUM.values() for m in [_re.match(r"(tables/[\w.\-]+\.md)", str(v.get("source", "")))] if m})
_missing = [c for c in _cited if not os.path.exists(os.path.join(exports.family_root(), c))]
assert not _missing, f"ledger sources cite tables that were not saved: {_missing}"
print(f"{len(_cited)} distinct tables cited by the ledger, all present.")
print({p: sum(k.startswith(p) for k in NUM) for p in ("ratio.", "ratio_floor.", "by_arm.", "by_iteration.", "step_multiplier.",
                                                        "sweep.", "sweep_top.", "crossjudge", "iso_channels", "api.", "api_ratio.",
                                                        "api_totals.", "figures.", "meta.")})

## 10 · Artifact index
Drop captions whose artifact no longer exists, then refresh `results/compute/INDEX.md` + `results/INDEX.md`.

In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())